In [1]:
import numpy as np
import torch
import torch.nn as nn
import time
import matplotlib.pyplot as plt
import os
import logging
from datetime import datetime
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm  # Use notebook-friendly tqdm

# Make sure your custom modules are available
from model import DeepONet
from utils import TrajectoryDataset, load_multi_traj_data, run_model_visualization

# Helper function from your script
def ellip_vol(model):
    d = model.V.log_diag_L.numel()
    c_val = model.c ** 2
    
    # Compute det(Q)^(-1/2)
    log_det_Q = 2 * torch.sum(model.V.log_diag_L)
    det_factor = torch.exp(-0.5 * log_det_Q)

    # Final volume
    if model.trainable_c:
        vol = (c_val**(d/2)) * det_factor
    else:
        vol = det_factor
    return vol

# --- Device Configuration ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(0)
np.random.seed(0)

print(f"Using device: {device}")

Using device: cuda


In [2]:
params = {
    'epochs': 10000,
    'bsize': 2048,
    'lam_reg_vol': 0.1,
    'project': True,
    'tag': '',
    'c_init': 1.0,
    'trainable_c': True,
    'trunk_scale': 0.05,
    'diag_Q': True,
    'output_dim': 256,
    'branch_conv_channels': [],
    'branch_fc_dims': [256],
    'trunk_hidden_dims': [256, 256, 256]
}

# --- Setup Directories ---
now = datetime.now()
save_time_str = now.strftime("%m%d_%H")
reg_name = ''
if params['trainable_c']: reg_name += 'cTrain'
if params['project']: reg_name += f'_proj_LamRegVol{params["lam_reg_vol"]}_C0{params["c_init"]}'
if params['diag_Q']: reg_name += '_diagQ'
    
save_name = f'E{params["epochs"]}_TS{params["trunk_scale"]}_branchConv{len(params["branch_conv_channels"])}_trunkHidden{len(params["trunk_hidden_dims"])}_{reg_name}_{params["tag"]}'
save_dir = os.path.join('Trained_Models', save_time_str, save_name)

params['save_dir'] = save_dir
model_folder = save_dir
figs_folder = os.path.join(save_dir, 'eval_results')

os.makedirs(model_folder, exist_ok=True)
os.makedirs(figs_folder, exist_ok=True)

print(f"Results will be saved in: {save_dir}")

Results will be saved in: Trained_Models/0818_15/E10000_TS0.05_branchConv0_trunkHidden3_cTrain_proj_LamRegVol0.1_C01.0_diagQ_


In [3]:
file_dir = 'Data/KS_data_batched_l100.53_grid512_M8_T500.0_dt0.01_amp5.0/data.npz'
data = np.load(file_dir, allow_pickle=True)

train_dataset, val_dataset = load_multi_traj_data(data, params['trunk_scale'])

train_loader = DataLoader(train_dataset, batch_size=params['bsize'], shuffle=True, pin_memory=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=params['bsize'], shuffle=False)

print(f"Created DataLoaders with {len(train_dataset)} training samples and {len(val_dataset)} validation samples.")

Created DataLoaders with 2994 training samples and 998 validation samples.


In [4]:
# Assuming u_batch is of shape (num_traj, traj_length, traj_dim)
m = s = data['u_batch'].shape[2]
n = 1

model_params = {
    'm': m,
    'n': n,
    'trainable_c': params['trainable_c'],
    'c0': params['c_init'],
    'project': params['project'],
    'diag_Q': params['diag_Q'],
    'branch_conv_channels': params['branch_conv_channels'],
    'branch_fc_dims': params['branch_fc_dims'],
    'trunk_hidden_dims': params['trunk_hidden_dims'],
    'output_dim': params['output_dim']
}

model = DeepONet(model_params).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
loss_func = torch.nn.MSELoss()

num_params = sum(v.numel() for v in model.parameters() if v.requires_grad)
print(f'Model initialized with {num_params:,} trainable parameters.')

Auto-detected flattened size for FC layer: 512
--- Initialized Branch Net Structure ---
Branch(
  (activation): ReLU()
  (conv_net): Sequential()
  (fc_net): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
  )
)

--- Initialized Trunk Net Structure ---
Trunk(
  (activation): ReLU()
  (net): Sequential(
    (0): Linear(in_features=1, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=256, bias=True)
    (5): ReLU()
    (6): Linear(in_features=256, out_features=256, bias=True)
  )
)
----------------------------------------
Projection layer included
V_elliptical initialized with a DIAGONAL Q.


Model initialized with 526,850 trainable parameters.


In [5]:
# --- IMPORTANT: Set this to the path of your checkpoint from epoch 6350 ---
# Note: Your original script seems to save the best model as 'model_epoch_best.pt'.
# If you have checkpoints for specific epochs, use that path here.
CHECKPOINT_PATH = "Trained_Models/debug_nan/model_epoch_best.pt"  # <--- 📝 CHANGE THIS

start_epoch = 0

if os.path.exists(CHECKPOINT_PATH):
    print(f"Loading checkpoint from: {CHECKPOINT_PATH}")
    # Based on your script, it saves the state_dict directly.
    # If you saved a dictionary with 'model_state_dict', 'optimizer_state_dict', etc.,
    # you will need to adjust this loading logic accordingly.
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
    
    # If your checkpoint also saved the optimizer state, uncomment the following:
    # checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    # model.load_state_dict(checkpoint['model_state_dict'])
    # optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    # start_epoch = checkpoint['epoch'] # Assuming you saved the epoch number
    
    # We will pretend we are at epoch 6350 to start debugging from the next step
    start_epoch = 6350
    
    print(f"✅ Checkpoint loaded. Ready to debug the step for epoch {start_epoch + 1}.")
else:
    print(f"⚠️ Checkpoint file not found at '{CHECKPOINT_PATH}'. Cannot proceed with debugging.")

Loading checkpoint from: Trained_Models/debug_nan/model_epoch_best.pt
✅ Checkpoint loaded. Ready to debug the step for epoch 6351.


In [6]:
# Enable PyTorch's anomaly detection to get a helpful stack trace when a NaN is created.
torch.autograd.set_detect_anomaly(True)

# Set the model to training mode
model.train()

# Get the very next batch of data that would be processed after your checkpoint
try:
    data_iter = iter(train_loader)
    x_batch, y_batch = next(data_iter)
    
    print("--- 🕵️‍♂️ Starting Debugging Step ---")

    # --- Step 1: Check inputs ---
    branch_batch, trunk_batch = x_batch
    print(f"Input branch shape: {branch_batch.shape}, Input trunk shape: {trunk_batch.shape}")
    if torch.isnan(branch_batch).any() or torch.isnan(trunk_batch).any():
        raise ValueError("NaN found in input data!")
    print("✅ Inputs are clean.")

    # Move data to device
    branch_batch = branch_batch.to(device)
    trunk_batch = trunk_batch.to(device)
    trunk_input = trunk_batch[0]
    y_batch = y_batch.to(device)

    # --- Step 2: Forward pass ---
    optimizer.zero_grad()
    outputs = model((branch_batch, trunk_input))
    
    # --- Step 3: Check model outputs ---
    print(f"Model outputs norm: {outputs.norm().item()}")
    if torch.isnan(outputs).any() or torch.isinf(outputs).any():
        raise ValueError("🔥 NaN or Inf detected in model output!")
    print("✅ Model outputs are clean.")

    # --- Step 4: Calculate loss ---
    dynamic_loss = loss_func(outputs, y_batch)
    if params['project']:
        vol = ellip_vol(model)
        reg_loss = params['lam_reg_vol'] * vol.squeeze()
    else:
        reg_loss = torch.tensor(0.0, device=device)
    loss = dynamic_loss + reg_loss

    # --- Step 5: Check loss value ---
    print(f"Calculated Loss: {loss.item():.6f} (Dynamic: {dynamic_loss.item():.6f}, Reg: {reg_loss.item():.6f})")
    if torch.isnan(loss).any() or torch.isinf(loss).any():
        raise ValueError("🔥 NaN or Inf detected in loss value!")
    print("✅ Loss is clean.")

    # --- Step 6: Backward pass (gradient calculation) ---
    loss.backward()
    print("✅ Backward pass completed.")

    # --- Step 7: Check gradients for NaNs ---
    grads_ok = True
    for name, param in model.named_parameters():
        if param.grad is not None:
            if torch.isnan(param.grad).any() or torch.isinf(param.grad).any():
                print(f"🔥🔥🔥 NaN or Inf detected in GRADIENTS of: {name}")
                grads_ok = False
    if grads_ok:
        print("✅ All gradients are clean.")

    # --- Step 8: Optimizer step (model weight update) ---
    optimizer.step()
    print("✅ Optimizer step completed.")
    
    # --- Step 9: Final check of model weights ---
    weights_ok = True
    for name, param in model.named_parameters():
        if torch.isnan(param.data).any() or torch.isinf(param.data).any():
            print(f"🔥🔥🔥 NaN or Inf detected in WEIGHTS of: {name} after optimizer step!")
            weights_ok = False
    if weights_ok:
        print("✅ All model weights are clean after update.")

    print("\n--- ✅ Debugging step finished successfully ---")
    
except Exception as e:
    print(f"\n--- ❌ ERROR during debugging step: {e} ---")
finally:
    # Disable anomaly detection after the debugging step
    torch.autograd.set_detect_anomaly(False)

--- 🕵️‍♂️ Starting Debugging Step ---
Input branch shape: torch.Size([2048, 512]), Input trunk shape: torch.Size([2048, 512, 1])
✅ Inputs are clean.
Model outputs norm: 1174.15380859375
✅ Model outputs are clean.
Calculated Loss: 0.204026 (Dynamic: 0.204025, Reg: 0.000001)
✅ Loss is clean.
✅ Backward pass completed.
✅ All gradients are clean.
✅ Optimizer step completed.
✅ All model weights are clean after update.

--- ✅ Debugging step finished successfully ---


In [7]:
# manually fix c
model.c.requires_grad = False

In [8]:
# --- Configuration for the Training Loop ---
epochs = params['epochs']
n_save_epochs = 50  # How often to evaluate, log, and save plots
best_loss = float('inf')

# Lists to store loss history for plotting
train_losses = []
val_losses = []
dynamic_losses = []
reg_losses = []

print(f"--- Starting/Resuming full training from epoch {start_epoch + 1} ---")
tic = time.time()

# --- Main Training Loop ---
for epoch in tqdm(range(start_epoch + 1, epochs + 1)):
	model.train()
	epoch_train_loss = 0
	epoch_dynamic_loss = 0
	epoch_reg_loss = 0
	
	# Iterate over batches from the DataLoader
	for x_batch, y_batch in train_loader:
		branch_batch, trunk_batch = x_batch
		
		# Move batch to the correct device
		branch_batch = branch_batch.to(device)
		trunk_batch = trunk_batch.to(device)
		trunk_input = trunk_batch[0]
		y_batch = y_batch.to(device)
		
		optimizer.zero_grad()
		
		# Forward pass
		u_pred = model((branch_batch, trunk_input))
		dynamic_loss = loss_func(u_pred, y_batch)

		# Calculate regularization loss if projection is enabled
		if params['project']:
			vol = ellip_vol(model)
			reg_loss = params['lam_reg_vol'] * vol.squeeze()
		else:
			reg_loss = torch.tensor(0.0, device=device)
		
		loss = dynamic_loss + reg_loss
		loss.backward()
		optimizer.step()
		
		# Accumulate losses for this epoch
		epoch_train_loss += loss.item()
		epoch_dynamic_loss += dynamic_loss.item()
		epoch_reg_loss += reg_loss.item()
				
	# Calculate average losses
	avg_train_loss = epoch_train_loss / len(train_loader)
	# avg_val_loss = epoch_val_loss / len(val_loader)
	avg_dynamic_loss = epoch_dynamic_loss / len(train_loader)
	avg_reg_loss = epoch_reg_loss / len(train_loader)
	print(f"Epoch {epoch}: Train Loss: {avg_train_loss:.6f}, Dynamic Loss: {avg_dynamic_loss:.6f}, Reg Loss: {avg_reg_loss:.6f}")

print("\n--- ✅ Training Complete ---")

--- Starting/Resuming full training from epoch 6351 ---


  0%|          | 0/3650 [00:00<?, ?it/s]

Epoch 6351: Train Loss: 1.568163, Dynamic Loss: 1.568162, Reg Loss: 0.000001
Epoch 6352: Train Loss: 0.979521, Dynamic Loss: 0.979520, Reg Loss: 0.000001
Epoch 6353: Train Loss: 0.796607, Dynamic Loss: 0.796605, Reg Loss: 0.000001
Epoch 6354: Train Loss: 0.517032, Dynamic Loss: 0.517030, Reg Loss: 0.000002
Epoch 6355: Train Loss: 0.503940, Dynamic Loss: 0.503938, Reg Loss: 0.000002
Epoch 6356: Train Loss: 0.515108, Dynamic Loss: 0.515106, Reg Loss: 0.000002
Epoch 6357: Train Loss: 0.433786, Dynamic Loss: 0.433783, Reg Loss: 0.000002
Epoch 6358: Train Loss: 0.370532, Dynamic Loss: 0.370529, Reg Loss: 0.000003
Epoch 6359: Train Loss: 0.353548, Dynamic Loss: 0.353545, Reg Loss: 0.000003
Epoch 6360: Train Loss: 0.355238, Dynamic Loss: 0.355234, Reg Loss: 0.000004
Epoch 6361: Train Loss: 0.339048, Dynamic Loss: 0.339043, Reg Loss: 0.000004
Epoch 6362: Train Loss: 0.315817, Dynamic Loss: 0.315812, Reg Loss: 0.000005
Epoch 6363: Train Loss: 0.301297, Dynamic Loss: 0.301291, Reg Loss: 0.000005

Exception ignored in: <function _releaseLock at 0x7e91a4849b80>
Traceback (most recent call last):
  File "/home/sunbochen/miniconda3/envs/chaos_env/lib/python3.9/logging/__init__.py", line 227, in _releaseLock
    def _releaseLock():
KeyboardInterrupt: 


RuntimeError: DataLoader worker (pid(s) 17908) exited unexpectedly

In [11]:
torch.min(model.V.log_diag_L)
model.c

Parameter containing:
tensor(0.8251, device='cuda:0')